In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install speechtokenizer

In [ ]:
import torch
import torchaudio
import numpy as np
from speechtokenizer import SpeechTokenizer
from huggingface_hub import hf_hub_download
from datasets import load_dataset, Audio
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, IterableDataset
from tqdm import tqdm
import os

In [ ]:
os.environ["HUGGINGFACE_TOKEN"] = "HF_TOKEN"

In [ ]:
# ===========================
# Configuration
# ===========================
MAX_T = 250  # Maximum time frame
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
SPEECHTOKENIZER_LR = 1e-5  # Lower learning rate for fine-tuning SpeechTokenizer
NUM_EPOCHS = 10
ENABLE_SPEECHTOKENIZER_FINETUNING = True

# Language to ID mapping
LANG_TO_ID = {
    "hindi": 0,
    "kannada": 1,
    "telugu": 2,
    "bengali": 3
}

In [ ]:
# ===========================
# Device Setup
# ===========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ===========================
# SpeechTokenizer Helper Functions
# ===========================
def load_speechtokenizer_model(device='cuda', hf_token=None, enable_grad=False):
    """Load SpeechTokenizer model and set requires_grad according to enable_grad."""
    print("Downloading SpeechTokenizer model files...")
    config_path = hf_hub_download(
        repo_id="fnlp/SpeechTokenizer",
        filename="speechtokenizer_hubert_avg/config.json",
        token=hf_token
    )

    ckpt_path = hf_hub_download(
        repo_id="fnlp/SpeechTokenizer",
        filename="speechtokenizer_hubert_avg/SpeechTokenizer.pt",
        token=hf_token
    )

    device = torch.device(device if torch.cuda.is_available() else "cpu")
    model = SpeechTokenizer.load_from_checkpoint(config_path, ckpt_path)

    # set requires_grad flags for all parameters according to enable_grad
    for p in model.parameters():
        p.requires_grad = bool(enable_grad)

    model.to(device)
    if enable_grad:
        model.train()
    else:
        model.eval()

    print("SpeechTokenizer loaded successfully (enable_grad=%s)!" % enable_grad)
    return model, device

def extract_speech_tokens(audio_array, original_sr, model, device, target_sr=16000, enable_grad=True):
    """Extract semantic and acoustic tokens from audio array - with gradient support"""
    # Convert to tensor and add batch dimension
    waveform = torch.tensor(audio_array, dtype=torch.float32).unsqueeze(0).to(device)
    
    # Resample if necessary
    if original_sr != target_sr:
        resampler = torchaudio.transforms.Resample(original_sr, target_sr).to(device)
        waveform = resampler(waveform)
    
    # Add channel dimension for SpeechTokenizer: [batch, time] -> [batch, 1, time]
    if waveform.dim() == 2:
        waveform = waveform.unsqueeze(1)
    
    # Extract tokens WITH gradients enabled for fine-tuning
    if enable_grad:
        encoded_result = model.encode(waveform)
    else:
        with torch.no_grad():
            encoded_result = model.encode(waveform)
    
    if encoded_result.dim() == 3:  # (n_q, B, T)
        semantic_tokens = encoded_result[:1, :, :]
        acoustic_tokens = encoded_result[1:, :, :] if encoded_result.shape[0] > 1 else None
    else:
        semantic_tokens = encoded_result
        acoustic_tokens = None
    
    return {
        'semantic_tokens': semantic_tokens,  # Keep on GPU with gradients
        'acoustic_tokens': acoustic_tokens,
    }

In [ ]:
def get_vectors_from_tokens(model, tokens, device, enable_grad=True):
    """Get embedding vectors from tokens - with gradient support for fine-tuning.
       Robust to acoustic_tokens==None.
    """
    codebook_layers = getattr(model.quantizer, 'vq', None)
    if codebook_layers is None:
        
        codebook_layers = getattr(model, 'quantizer', None)

    retrieved_vectors = {}

    
    semantic_indices = tokens.get('semantic_tokens', None)
    if semantic_indices is None:
        raise RuntimeError("No semantic tokens found.")

    semantic_indices = semantic_indices.to(device)

    try:
        semantic_codebook = model.quantizer.vq.layers[0]._codebook.embed.to(device)
        if enable_grad:
            semantic_vectors = semantic_codebook[semantic_indices]
        else:
            with torch.no_grad():
                semantic_vectors = semantic_codebook[semantic_indices]
    except Exception:
        
        with torch.no_grad():
            semantic_vectors = semantic_indices.float()

    retrieved_vectors['semantic_vectors'] = semantic_vectors.squeeze(0)

    
    acoustic_indices = tokens.get('acoustic_tokens', None)
    acoustic_vectors_list = []
   
    if True:
        acoustic_indices = acoustic_indices.to(device)
        num_acoustic_quantizers = acoustic_indices.shape[0]
            
        for i in range(num_acoustic_quantizers):
            indices_level = acoustic_indices[i, :, :]
            try:
                acoustic_codebook = model.quantizer.vq.layers[i + 1]._codebook.embed.to(device)
                if enable_grad:
                    vectors_level = acoustic_codebook[indices_level]
                else:
                    with torch.no_grad():
                        vectors_level = acoustic_codebook[indices_level]
            except Exception:
                # fallback: use semantic vectors as placeholder
                vectors_level = retrieved_vectors['semantic_vectors'].unsqueeze(0)
            acoustic_vectors_list.append(vectors_level)
    retrieved_vectors['acoustic_vectors'] = torch.stack(acoustic_vectors_list)
    return retrieved_vectors

In [ ]:
class KathbathDataset(IterableDataset):

    def __init__(self, languages, split="train", tokenizer_model=None, device='cuda',
                 max_t=None, fallback_max_t=MAX_T, enable_grad=True, peek_n=40):
        self.languages = languages
        self.split = split
        self.tokenizer_model = tokenizer_model
        self.device = device
        self.enable_grad = enable_grad
        self.fallback_max_t = fallback_max_t
        self.peek_n = peek_n

        # map language -> dataset iterator and a small buffer (for re-yielding peeked examples)
        self.lang_iters = {}
        self.buffers = {}
        self.lang_ds_objs = {}

        # load dataset objects (streaming)
        for lang in languages:
            ds = load_dataset("ai4bharat/Kathbath", lang, split=split, streaming=True)
            ds = ds.cast_column("audio_filepath", Audio(sampling_rate=16000))
            self.lang_ds_objs[lang] = ds
            self.lang_iters[lang] = iter(ds)
            self.buffers[lang] = []

        # estimate max_t if not provided
        if max_t is None:
            print(f"Estimating max_t by peeking up to {self.peek_n} examples per language (this may take time).")
            max_found = 0
            for lang in languages:
                it = self.lang_iters[lang]
                buf = self.buffers[lang]
                for i in range(self.peek_n):
                    try:
                        ex = next(it)
                        buf.append(ex)  # buffer so we can re-use it later
                        # extract tokens quickly without enabling grad to speed up peeking
                        try:
                            audio_array = ex["audio_filepath"]["array"]
                            sampling_rate = ex["audio_filepath"]["sampling_rate"]
                            tokens = extract_speech_tokens(audio_array, sampling_rate, self.tokenizer_model, self.device, enable_grad=False)
                            # semantic tokens shape: [1, B, T] or [B, T, D] depending on tokenizer
                            sem = tokens.get('semantic_tokens', None)
                            ac = tokens.get('acoustic_tokens', None)
                            tlen = 0
                            if sem is not None:
                                # support both possible shapes
                                if sem.dim() == 3:
                                    tlen = sem.shape[-1]
                                elif sem.dim() == 2:
                                    tlen = sem.shape[1]
                            elif ac is not None:
                                # use acoustic time axis if semantic missing
                                if ac.dim() == 4:  # [n_q, B, T, D]
                                    tlen = ac.shape[2]
                                else:
                                    tlen = ac.shape[-2]
                            max_found = max(max_found, tlen)
                        except Exception:
                            continue
                    except StopIteration:
                        break
                    except Exception:
                        break
            if max_found <= 0:
                print("Could not detect max_t from peeked examples. Falling back to", fallback_max_t)
                self.max_t = fallback_max_t
            else:
                # clamp to fallback if too large
                self.max_t = min(max_found, fallback_max_t)
                print(f"Detected max token length {max_found}. Using max_t={self.max_t}")
            # reconstruct iterators: for each language create an iterator that yields buffer first then remaining stream
            for lang in languages:
                def make_iter(lang):
                    # yield from buffer then rest of original iterator
                    for ex in self.buffers[lang]:
                        yield ex
                    for ex in self.lang_ds_objs[lang]:
                        yield ex
                self.lang_iters[lang] = make_iter(lang)
        else:
            self.max_t = max_t

    def __iter__(self):
        # yield across languages in round-robin fashion
        while True:
            made_progress = False
            for lang in self.languages:
                it = self.lang_iters[lang]
                try:
                    example = next(it)
                except StopIteration:
                    continue
                except TypeError:
                    # if it's an hg dataset Iterable, convert to iterator then next
                    it2 = iter(self.lang_ds_objs[lang])
                    self.lang_iters[lang] = it2
                    try:
                        example = next(it2)
                    except Exception:
                        continue

                made_progress = True
                lang_id = LANG_TO_ID.get(lang, 0)
                speaker_id = example.get('speaker_id', None)
                try:
                    audio_array = example["audio_filepath"]["array"]
                    sampling_rate = example["audio_filepath"]["sampling_rate"]
                except Exception as e:
                    # malformed example - skip
                    print(f"Skipping example in {lang} due to audio read error: {e}")
                    continue

                # extract tokens and retrieve vectors (enable_grad controls tokenizer grad)
                try:
                    tokens = extract_speech_tokens(audio_array, sampling_rate, self.tokenizer_model, self.device, enable_grad=self.enable_grad)
                    vectors = get_vectors_from_tokens(self.tokenizer_model, tokens, self.device, enable_grad=self.enable_grad)
                except Exception as e:
                    # if tokenization fails skip
                    print(f"Tokenization failed for an example: {e}")
                    continue

                semantic_vectors = vectors.get('semantic_vectors', None)  # [B, T, D] expected
                acoustic_vectors = vectors.get('acoustic_vectors', None)  # [n_q, B, T, D] expected

                # guard shapes
                if semantic_vectors is None:
                    # if semantic missing try to derive some placeholder from acoustic
                    if acoustic_vectors is not None:
                        # sum quantizers -> [B, T, D]
                        semantic_vectors = torch.sum(acoustic_vectors, dim=0)
                    else:
                        continue

                # normalize expected dims: ensure shapes [B, T, D]
                if semantic_vectors.dim() == 3:
                    pass
                elif semantic_vectors.dim() == 2:
                    semantic_vectors = semantic_vectors.unsqueeze(0)
                else:
                    # unexpected -> skip
                    continue

                if acoustic_vectors is None:
                    # create zeros placeholder of shape [1, semantic_T, D]
                    acoustic_vectors = semantic_vectors.unsqueeze(0)  # [1, B, T, D]
                elif acoustic_vectors.dim() == 4:
                    pass
                else:
                    # try to reshape
                    acoustic_vectors = acoustic_vectors.unsqueeze(0)

                B = semantic_vectors.shape[0]
                seq_len_sem = semantic_vectors.shape[1]
                seq_len_ac = acoustic_vectors.shape[2]
                max_time_frames = min(seq_len_sem, seq_len_ac, self.max_t)

                # waveform on device
                waveform = torch.tensor(audio_array, dtype=torch.float32).to(self.device)

                # yield per time frame
                for t in range(max_time_frames):
                    # summed acoustic across quantizers: [B, T, D]
                    summed_acoustic = torch.sum(acoustic_vectors, dim=0).to(self.device)
                    spk_embedding = summed_acoustic[0, t, :].to(self.device)
                    lang_embedding = semantic_vectors[0, t, :].to(self.device)

                    yield {
                        'waveform': waveform,                 # ON GPU
                        'spk_embedding': spk_embedding,      # ON GPU
                        'lang_embedding': lang_embedding,    # ON GPU
                        'speaker_id': speaker_id,
                        'lang_id': lang_id,
                        'time_frame': t
                    }

            if not made_progress:
                break  # all iterators exhausted

In [ ]:
def collate_fn(batch):
    """Custom collate function to handle variable length audio - KEEP EVERYTHING ON GPU"""
    # Get device from first item (all items should already be on GPU)
    device = batch[0]['waveform'].device
    
    # Find max waveform length in batch
    max_len = max([item['waveform'].shape[0] for item in batch])
    
    # Pad waveforms (already on GPU)
    waveforms = []
    for item in batch:
        waveform = item['waveform']
        if waveform.shape[0] < max_len:
            padding = torch.zeros(max_len - waveform.shape[0], device=device)
            waveform = torch.cat([waveform, padding])
        waveforms.append(waveform)
    
    return {
        'waveforms': torch.stack(waveforms),  # Already on GPU
        'spk_embeddings': torch.stack([item['spk_embedding'] for item in batch]),  # Already on GPU
        'lang_embeddings': torch.stack([item['lang_embedding'] for item in batch]),  # Already on GPU
        'speaker_ids': torch.tensor([hash(item['speaker_id']) % 1000 for item in batch], device=device),
        'lang_ids': torch.tensor([item['lang_id'] for item in batch], device=device),
        'time_frames': [item['time_frame'] for item in batch]
    }


In [ ]:
# ===========================
# Model Components 
# ===========================
class CrossFeaturePrefixTuner(nn.Module):
    def __init__(self, dim: int, num_heads: int = 4, prefix_len: int = 5):
        super().__init__()
        self.prefix_k = nn.Parameter(torch.randn(prefix_len, dim) * 0.01)
        self.prefix_v = nn.Parameter(torch.randn(prefix_len, dim) * 0.01)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)

    def forward(self, query: torch.Tensor, kv: torch.Tensor) -> torch.Tensor:
        B = kv.size(0)
        prefix_k = self.prefix_k.unsqueeze(0).expand(B, -1, -1).to(kv.device)
        prefix_v = self.prefix_v.unsqueeze(0).expand(B, -1, -1).to(kv.device)
        
        k = torch.cat([prefix_k, kv], dim=1)
        v = torch.cat([prefix_v, kv], dim=1)
        
        out, _ = self.attn(query, k, v)
        out = self.norm(out)
        return out


In [ ]:
class DecoderLSTMFCUp(nn.Module):
    def __init__(self, embed_dim: int, n_mels: int = 80, hidden_dim: int = 512,
                 lstm_layers: int = 2, dropout: float = 0.1):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.n_mels = n_mels
        
        self.pre = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        
        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
            bidirectional=False,
        )
        
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_mels),
        )
        
        self.out_ln = nn.LayerNorm(n_mels)

    def forward(self, query: torch.Tensor, out_len: int):
        B, seq_len, _ = query.size()
        
        if seq_len < out_len:
            repeat_factor = (out_len + seq_len - 1) // seq_len
            query = query.repeat(1, repeat_factor, 1)
            query = query[:, :out_len, :]
        elif seq_len > out_len:
            query = query[:, :out_len, :]
        
        x = self.pre(query)
        lstm_out, _ = self.lstm(x)
        out = self.head(lstm_out)
        out = self.out_ln(out)
        return out


In [ ]:
class AAMSoftmax(nn.Module):
    def __init__(self, n_classes, feat_dim, s=30.0, m=0.2):
        super(AAMSoftmax, self).__init__()
        self.n_classes = n_classes
        self.feat_dim = feat_dim
        self.s = s
        self.m = m
        self.eps = 1e-7
        
        self.weight = nn.Parameter(torch.randn(n_classes, feat_dim) * 0.01)
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, labels):
        if torch.isnan(x).any():
            x = torch.nan_to_num(x, nan=0.0)
        
        x_norm = F.normalize(x, p=2, dim=1, eps=self.eps)
        w_norm = F.normalize(self.weight, p=2, dim=1, eps=self.eps)
        
        cosine = F.linear(x_norm, w_norm)
        cosine = torch.clamp(cosine, -1.0 + self.eps, 1.0 - self.eps)
        
        phi = cosine - self.m
        
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1).long(), 1)
        
        output = (one_hot * phi) + ((1.0 - one_hot) * cosine)
        output *= self.s
        
        return F.cross_entropy(output, labels)

In [ ]:
class LASPA(nn.Module):
    def __init__(self, spk_dim: int, lang_dim: int, proj_dim: int = 768, 
                 num_heads: int = 4, prefix_len: int = 5, n_mels: int = 80, 
                 hidden_dim: int = 512, num_speakers: int = 1000, num_languages: int = 4):
        super().__init__()
        
        self.lang_proj_layer = nn.Sequential(
            nn.Linear(lang_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.Dropout(0.1)
        )
        self.spk_proj_layer = nn.Sequential(
            nn.Linear(spk_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.Dropout(0.1)
        )
        
        self.spk2lang = CrossFeaturePrefixTuner(proj_dim, num_heads=num_heads, prefix_len=prefix_len)
        self.lang2spk = CrossFeaturePrefixTuner(proj_dim, num_heads=num_heads, prefix_len=prefix_len)
        
        self.decoder = DecoderLSTMFCUp(embed_dim=2 * proj_dim, n_mels=n_mels, hidden_dim=hidden_dim)
        
        self.mel_tf = torchaudio.transforms.MelSpectrogram(
            sample_rate=16000,
            n_fft=400,
            win_length=400,
            hop_length=160,
            f_min=0.0,
            f_max=8000.0,
            n_mels=n_mels,
            center=True,
            pad_mode="reflect",
            power=2.0,
            norm=None,
            mel_scale="htk"
        ).to(device)
        
        self.speaker_classifier = AAMSoftmax(num_speakers, proj_dim, s=30.0, m=0.2)
        
        self.language_classifier = nn.Sequential(
            nn.LayerNorm(proj_dim),
            nn.Linear(proj_dim, num_languages)
        )
        nn.init.xavier_uniform_(self.language_classifier[1].weight, gain=0.1)
        nn.init.zeros_(self.language_classifier[1].bias)

    @staticmethod
    def _log_mel(mel: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
        return torch.log(mel + eps)

    def forward(self, waveforms, spk_embeddings, lang_embeddings, speaker_ids=None, language_ids=None):
        # Project embeddings
        spk_proj = self.spk_proj_layer(spk_embeddings)
        lang_proj = self.lang_proj_layer(lang_embeddings)
        
        # Prepare for cross-attention
        spk_q = spk_proj.unsqueeze(1)
        lang_q = lang_proj.unsqueeze(1)
        
        # Cross-feature fusion
        spk2lang_out = self.spk2lang(spk_q, lang_q)
        lang2spk_out = self.lang2spk(lang_q, spk_q)
        
        # Concatenate fused features
        fused = torch.cat([spk2lang_out, lang2spk_out], dim=-1)
        
        # Generate mel spectrogram target
        with torch.no_grad():
            mel = self.mel_tf(waveforms.to(device))
            mel = self._log_mel(mel)
            mel_target = mel.transpose(1, 2)
        
        out_len = mel_target.size(1)
        
        # Decode to mel spectrogram
        mel_hat = self.decoder(fused, out_len=out_len)
        
        # Compute classification losses
        speaker_loss = None
        language_logits = None
        
        if speaker_ids is not None:
            speaker_loss = self.speaker_classifier(spk_proj, speaker_ids)
        
        if language_ids is not None:
            language_logits = self.language_classifier(lang_proj)
        
        return mel_hat, mel_target, spk_proj, lang_proj, speaker_loss, language_logits

In [ ]:
# ===========================
# Loss Functions
# ===========================
def compute_lmse(recon: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    loss = F.mse_loss(recon, target)
    if torch.isnan(loss):
        return torch.tensor(0.0, device=recon.device)
    return loss

def compute_lmapc(spk_emb: torch.Tensor, lang_emb: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    spk = spk_emb - spk_emb.mean(dim=-1, keepdim=True)
    lang = lang_emb - lang_emb.mean(dim=-1, keepdim=True)
    
    num = (spk * lang).sum(dim=-1)
    den = torch.norm(spk, dim=-1) * torch.norm(lang, dim=-1) + eps
    corr = num / den
    
    loss = corr.abs().mean()
    
    if torch.isnan(loss):
        return torch.tensor(0.0, device=spk_emb.device)
    
    return loss

In [ ]:
class LASPALoss(nn.Module):
    def __init__(self, alpha: float = 0.01, beta: float = 0.1, gamma: float = 10, delta: float = 10):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.delta = delta

    def forward(self, mel_hat, mel_target, spk_emb, lang_emb, speaker_loss, language_logits, language_ids):
        lmse = compute_lmse(mel_hat, mel_target)
        lmapc = compute_lmapc(spk_emb, lang_emb)
        
        aam_loss = torch.tensor(0.0, device=mel_hat.device)
        nll_loss = torch.tensor(0.0, device=mel_hat.device)
        
        if speaker_loss is not None:
            aam_loss = speaker_loss
            if torch.isnan(aam_loss):
                aam_loss = torch.tensor(0.0, device=mel_hat.device)
        
        if language_logits is not None and language_ids is not None:
            if torch.isnan(language_logits).any():
                language_logits = torch.nan_to_num(language_logits, nan=0.0)
            
            language_logits = torch.clamp(language_logits, min=-100, max=100)
            nll_loss = F.cross_entropy(language_logits, language_ids)
            
            if torch.isnan(nll_loss):
                nll_loss = torch.tensor(0.0, device=mel_hat.device)
        
        total = self.alpha * lmse + self.beta * lmapc + self.gamma * aam_loss + self.delta * nll_loss
        
        if torch.isnan(total):
            total = lmse
        
        metrics = {
            "LMSE": float(lmse.detach().cpu()),
            "LMAPC": float(lmapc.detach().cpu()),
            "AAM": float(aam_loss.detach().cpu()),
            "NLL": float(nll_loss.detach().cpu())
        }
        
        return total, metrics


In [ ]:
# ===========================
# Training Loop 
# ===========================
def train():
    # Load SpeechTokenizer with gradient support
    tokenizer_model, _ = load_speechtokenizer_model(
    device='cuda',
    hf_token=os.environ.get("HUGGINGFACE_TOKEN", None),
    enable_grad=ENABLE_SPEECHTOKENIZER_FINETUNING
)


    languages = ["hindi", "kannada", "telugu", "bengali"]
    dataset = KathbathDataset(
        languages=languages,
        split="train",
        tokenizer_model=tokenizer_model,
        device=device,
        max_t=MAX_T,
        enable_grad=ENABLE_SPEECHTOKENIZER_FINETUNING
    )

    dataloader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        collate_fn=collate_fn,
        num_workers=0  # always 0 for IterableDataset
    )

    # Get embedding dimensions from first batch
    first_batch = next(iter(dataloader))
    spk_dim = first_batch['spk_embeddings'].shape[-1]
    lang_dim = first_batch['lang_embeddings'].shape[-1]

    # Initialize LASPA model
    model = LASPA(
        spk_dim=spk_dim,
        lang_dim=lang_dim,
        proj_dim=768,
        num_heads=4,
        prefix_len=5,
        n_mels=80,
        hidden_dim=512,
        num_speakers=1000,
        num_languages=len(LANG_TO_ID)
    ).to(device)

    criterion = LASPALoss(alpha=0.01, beta=0.1, gamma=10, delta=10)

    if ENABLE_SPEECHTOKENIZER_FINETUNING:
        tokenizer_params = [p for p in tokenizer_model.parameters() if p.requires_grad]
        laspa_params = list(model.parameters())
        optimizer = torch.optim.AdamW([
            {'params': tokenizer_params, 'lr': SPEECHTOKENIZER_LR},
            {'params': laspa_params, 'lr': LEARNING_RATE}
        ])
    else:
        for p in tokenizer_model.parameters():
            p.requires_grad = False
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    for epoch in range(NUM_EPOCHS):
        model.train()
        if ENABLE_SPEECHTOKENIZER_FINETUNING:
            tokenizer_model.train()

        epoch_losses = []
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

        for batch_idx, batch in enumerate(pbar):
            try:
                waveforms = batch['waveforms']
                spk_embeddings = batch['spk_embeddings']
                lang_embeddings = batch['lang_embeddings']
                speaker_ids = batch['speaker_ids']
                lang_ids = batch['lang_ids']

                mel_hat, mel_target, spk_proj, lang_proj, speaker_loss, language_logits = model(
                    waveforms, spk_embeddings, lang_embeddings, speaker_ids, lang_ids
                )

                loss, metrics = criterion(
                    mel_hat, mel_target, spk_proj, lang_proj,
                    speaker_loss, language_logits, lang_ids
                )

                optimizer.zero_grad()
                loss.backward()

                if ENABLE_SPEECHTOKENIZER_FINETUNING:
                    torch.nn.utils.clip_grad_norm_(tokenizer_model.parameters(), max_norm=1.0)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)

                optimizer.step()

                epoch_losses.append(loss.item())
                pbar.set_postfix({
                    'loss': f"{loss.item():.4f}",
                    'LMSE': f"{metrics['LMSE']:.4f}",
                    'AAM': f"{metrics['AAM']:.4f}",
                    'NLL': f"{metrics['NLL']:.4f}",
                    'lr': f"{optimizer.param_groups[0]['lr']:.2e}"
                })

                if batch_idx % 1000 == 0 and batch_idx > 0:
                    checkpoint = {
                        'epoch': epoch,
                        'batch_idx': batch_idx,
                        'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'scheduler_state_dict': scheduler.state_dict(),
                        'loss': loss.item(),
                    }
                    if ENABLE_SPEECHTOKENIZER_FINETUNING:
                        checkpoint['tokenizer_state_dict'] = tokenizer_model.state_dict()
                    torch.save(checkpoint, f'checkpoint_epoch{epoch}_step{batch_idx}.pt')

            except Exception as e:
                print(f"\n✗ Error in batch {batch_idx}: {e}")
                continue

        scheduler.step()
        avg_loss = np.mean(epoch_losses) if epoch_losses else 0
        print(f"\nEpoch {epoch+1} Summary: Avg Loss {avg_loss:.4f}")

        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'loss': avg_loss,
        }
        if ENABLE_SPEECHTOKENIZER_FINETUNING:
            checkpoint['tokenizer_state_dict'] = tokenizer_model.state_dict()
        torch.save(checkpoint, f'checkpoint_epoch{epoch+1}.pt')

    final_checkpoint = {
        'model_state_dict': model.state_dict(),
        'config': {
            'spk_dim': spk_dim,
            'lang_dim': lang_dim,
            'num_speakers': 1000,
            'num_languages': len(LANG_TO_ID)
        }
    }
    if ENABLE_SPEECHTOKENIZER_FINETUNING:
        final_checkpoint['tokenizer_state_dict'] = tokenizer_model.state_dict()
    torch.save(final_checkpoint, 'final_model.pt')
    print("✓ Final model saved as 'final_model.pt'")


In [ ]:
if __name__ == "__main__":
    train()